# 8. Compress adjusted fields
This notebook
* writes compressed copies of the adjusted rainfall fields from `3_adjust_radar` for archiving on Zenodo
* adds metadata describing the method, source data, units and license
* uses lossless compression (zlib with shuffle), so all values are identical to the originals
* verifies that the compressed files contain exactly the same data as the originals

The compressed files can be placed in `data/adjusted_fields/` and read by the other notebooks without changes.

In [1]:
import glob
import os

import xarray as xr

In [2]:
input_path = 'data/adjusted_fields/'
output_path = 'data/adjusted_fields_compressed/'
os.makedirs(output_path, exist_ok=True)

license = 'CC BY-SA 4.0'
creator = 'Erlend Øydvin, Elia Covi, Maximilian Graf, Christian Chwala'
references = (
    'https://doi.org/10.5194/egusphere-2025-6371 ; '
    'https://github.com/OpenSenseAction/radar_adjustment_intercomparison'
)

In [3]:
# Merging methods, acronyms as in Table 1 of the paper
methods = {
    'add_IDW': ('add_p_idw', 'additive merging with inverse distance weighting'),
    'add_pBK': ('add_p_ok', 'additive merging with ordinary kriging, CMLs as points'),
    'add_lBK': ('add_b_ok', 'additive merging with ordinary block kriging, CMLs as blocks'),
    'mul_IDW': ('mul_p_idw', 'multiplicative merging with inverse distance weighting'),
    'mul_pBK': ('mul_p_ok', 'multiplicative merging with ordinary kriging, CMLs as points'),
    'mul_lBK': ('mul_b_ok', 'multiplicative merging with ordinary block kriging, CMLs as blocks'),
    'pked': ('ked_p', 'kriging with external drift, CMLs as points'),
    'lked': ('ked_b', 'kriging with external drift, CMLs as blocks'),
}

datasets = {
    'OpenMRG': 'OpenMRG v1.1 (Andersson et al., 2022), https://zenodo.org/records/7107689',
    'OpenRainER': 'OpenRainER v2.0.0 (Covi and Roversi, 2025), https://zenodo.org/records/14731404',
}

In [4]:
def add_metadata(ds, dataset, method):
    acronym, description = methods[method]
    ds.attrs = {
        'title': f'Radar rainfall merged with CML data, {dataset}, {acronym}',
        'summary': (
            f'Hourly radar rainfall fields adjusted with commercial microwave link (CML) data '
            f'using {description}. Produced for Øydvin et al. (2026), Hydrol. Earth Syst. Sci.'
        ),
        'method_acronym': acronym,
        'source': datasets[dataset],
        'creator_name': creator,
        'references': references,
        'history': 'Created by 3_adjust_radar.ipynb, compressed by 8_compress_adjusted_fields.ipynb',
        'license': license,
    }

    # Replace attributes inherited from the radar data
    ds['rainfall'].attrs = {
        'long_name': 'hourly rainfall accumulation',
        'units': 'mm',
        'cell_methods': 'time: sum',
    }
    ds['time'].attrs['comment'] = 'End of the 1-hour accumulation interval'
    ds['lon'].attrs = {'long_name': 'longitude of grid cell center', 'units': 'degrees_east'}
    ds['lat'].attrs = {'long_name': 'latitude of grid cell center', 'units': 'degrees_north'}
    for coord in ['x_grid', 'y_grid']:
        ds[coord].attrs = {'long_name': f'{coord[0]} coordinate of grid cell center in UTM zone 32N', 'units': 'm', 'projection': 'EPSG:32632'}
    if dataset == 'OpenMRG':
        for coord in ['x', 'y']:
            ds[coord].attrs['units'] = 'm'
            ds[coord].attrs['proj4'] = '+proj=stere +lat_ts=60 +ellps=bessel +lon_0=14 +lat_0=90'
    return ds

In [5]:
# Compress all adjusted fields, one time step per chunk
for fn in sorted(glob.glob(input_path + '*.nc')):
    fn_out = output_path + os.path.basename(fn)
    dataset, method = os.path.basename(fn)[:-3].split('_rainfall_')
    with xr.open_dataset(fn) as ds:
        ds = ds.load()
    ds_out = add_metadata(ds.copy(deep=True), dataset, method)
    encoding = {
        'rainfall': {
            'zlib': True,
            'complevel': 4,
            'shuffle': True,
            'chunksizes': (1, ds.sizes['y'], ds.sizes['x']),
        }
    }
    ds_out.to_netcdf(fn_out, encoding=encoding)

    # Check that the compressed file has the same values as the original
    with xr.open_dataset(fn_out) as ds_check:
        assert ds.equals(ds_check.load()), fn

    size_in = os.path.getsize(fn) / 1e6
    size_out = os.path.getsize(fn_out) / 1e6
    print(f'{os.path.basename(fn)}: {size_in:.0f} MB -> {size_out:.0f} MB')

OpenMRG_rainfall_add_IDW.nc: 31 MB -> 5 MB
OpenMRG_rainfall_add_lBK.nc: 32 MB -> 5 MB
OpenMRG_rainfall_add_pBK.nc: 32 MB -> 5 MB
OpenMRG_rainfall_lked.nc: 31 MB -> 5 MB
OpenMRG_rainfall_mul_IDW.nc: 31 MB -> 6 MB
OpenMRG_rainfall_mul_lBK.nc: 31 MB -> 6 MB
OpenMRG_rainfall_mul_pBK.nc: 31 MB -> 6 MB
OpenMRG_rainfall_pked.nc: 31 MB -> 5 MB
OpenRainER_rainfall_add_IDW.nc: 1912 MB -> 77 MB
OpenRainER_rainfall_add_lBK.nc: 1912 MB -> 75 MB
OpenRainER_rainfall_add_pBK.nc: 1912 MB -> 75 MB
OpenRainER_rainfall_lked.nc: 1912 MB -> 85 MB
OpenRainER_rainfall_mul_IDW.nc: 1912 MB -> 90 MB
OpenRainER_rainfall_mul_lBK.nc: 1912 MB -> 90 MB
OpenRainER_rainfall_mul_pBK.nc: 1912 MB -> 90 MB
OpenRainER_rainfall_pked.nc: 1912 MB -> 85 MB
